# Gold Layer — dim_restaurant (SCD 2) + dim_dish + fact_video_restaurant_mention

Source: `silver.stg_restaurant_mentions` (LLM extraction), Google (`stg_restaurant_geocode`, `stg_restaurant_place_details`, `stg_places`), OpenStreetMap. Run **after** `gold/dims` (needs `dim_video`).

**Key (v2, 2026-11-02, datle): no surrogate key.** The dimension is keyed by the durable `restaurant_id` = `md5('place:' || place_id)` plus `valid_from` (one row per SCD 2 version). Facts store `restaurant_id`; the current version is `is_current`, the version at a time T is `valid_from <= T < valid_to`. Unmatched mentions use `restaurant_id = 'unknown'` (one Unknown row in the dimension). The old `restaurant_key` column is dropped (one-time migration below keeps every version).

Only restaurants with a Google `place_id` enter the dimension: video mentions once `stg_restaurant_geocode` matched them, Google-only places from `stg_places`. Name, address or attribute changes never change the id.

**Address with GSO (TCTK) codes** (reference `src/reference/vn_new_ward_to_old_district.csv`, `vn_old_districts.csv`, built by `scripts/build_admin_reference.py` from the `vietnamadminunits` package):
- `address_full` (Google formatted address), `street` (the part before the ward).
- `ward_code` / `ward_name`: the **new** ward (after 01/07/2025), from Google's ward name in the province; else the nearest new-ward centroid (`ward_source` = `google` / `nearest`).
- `district_code` / `district_name`: the **old** district (before 2025; districts no longer exist), first found of: Google `addressComponents` → the district said in the video → the new ward's old district when the ward came from one old district only → NULL. `district_source` = `google` / `video` / `ward`. `districts_old` lists every old district the ward came from (17 of 168 HCMC wards span 2).
- `city_code` / `city_name`: province, e.g. `79` / `Thành phố Hồ Chí Minh`.

**SCD 2** (tracked: `address_full`, `ward_code`, `district_code`, `city_code`, latitude / longitude, `place_id`): a change closes the current version (`valid_to = run time`, `is_current = false`) and inserts a new one; a restaurant no longer in the source is closed. Everything else is **type 1** (overwrites every version).

**Type 1 attributes:**
- `restaurant_type`: Google primary type (cafe, bar, bakery, fast food, buffet, food court), else words in the name (`nhà hàng` → Nhà hàng; `quán`, `xe`, `tiệm`, phở / bún / cơm ... → Quán ăn), else a Google restaurant type (Nhà hàng when the price band is 300k+, else Quán ăn), else `Chưa rõ`.
- `business_status` (Google: `OPERATIONAL` / `CLOSED_TEMPORARILY` / `CLOSED_PERMANENTLY`) and `is_active` (false only when permanently closed). No closing date (datle).
- Opening hours: `open_time` / `close_time` (`HH:mm`, earliest opening and latest closing of the week), `is_open_24h`, `opening_hours_text`, `hours_source` = `google` (Place Details `regularOpeningHours.periods`) / `video` (said in the review) / `osm`.
- Cuisine (`cuisine`, `cuisine_detail`, `cuisine_source`: Google specific type > keyword in the name > OSM > Google generic), price (`price_band`, `price_min_vnd` / `price_max_vnd`, `price_source` = `google` / `video`), `google_maps_uri`, `photo_name`, `google_website`, `google_phone`, `osm_key`, `has_youtube_review`.
- From the videos: `signature_dishes`, `menu_dishes`, `video_price_min_vnd` / `video_price_max_vnd` (details per dish in `dim_dish`).

No rating in the dimension: ratings are measures in `fact_place_rating_snapshot` (`gold/place_rating`).

**dim_dish** (SCD 1, full rebuild, child of `dim_restaurant`): one row per `restaurant_id` × `dish_name_normalized` (dish name lower-cased, diacritics kept) from the dishes the reviews name (`stg_restaurant_mentions.dishes`, extraction v2). `dish_name` (longest spelling), `dish_category`, `price_vnd` (median price said, 5,000–5,000,000 VND), `price_min_vnd` / `price_max_vnd`, `is_signature` (a review called it the signature dish), `video_count`, `first_seen_at` / `last_seen_at`, `source` = `video`.

**fact_video_restaurant_mention:** factless bridge, grain `video_key` × `restaurant_id`. `mention_weight = 1 / restaurants in the video` (sums to 1 per video; restaurants not geocoded yet share one `'unknown'` row per video). Full-sync MERGE (atomic).

**dim_google_review** (SCD 1, full rebuild): one row per Google review, `review_key = xxhash64(review_id)` (−1 = Unknown), `restaurant_id` of the place (`'unknown'` if none).

OpenStreetMap match (cuisine / opening hours gaps): nearest OSM place within 100 m with a similar name. Data © OpenStreetMap contributors (ODbL).

In [ ]:
%run ./_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

DIM = f"{GOLD}.dim_restaurant"
DISH = f"{GOLD}.dim_dish"
FACT = f"{GOLD}.fact_video_restaurant_mention"
GEOCODE = f"{SILVER}.stg_restaurant_geocode"  # Google Places lookups (stg/restaurant_geocode)
DETAILS = f"{SILVER}.stg_restaurant_place_details_latest"  # Google Place Details, latest read per place (view, stg/restaurant_place_details)
PLACES = f"{SILVER}.stg_places_latest"  # Google Maps Text Search, latest read of open food places (view, stg/places), Google-only restaurants
OSM = f"{SILVER}.stg_osm_restaurants"  # OpenStreetMap food places (stg/osm_restaurants), fills cuisine / opening hours
REVIEWS = f"{SILVER}.stg_restaurant_place_reviews"  # Google reviews (stg/restaurant_place_details)
REVIEW_DIM = f"{GOLD}.dim_google_review"
OSM_MAX_DISTANCE_M = 100
for src_table, nb_name in [(GEOCODE, "stg/restaurant_geocode"), (DETAILS, "stg/restaurant_place_details")]:
    if not spark.catalog.tableExists(src_table):
        raise RuntimeError(f"{src_table} does not exist: run {nb_name} first")
if not spark.catalog.tableExists(PLACES):  # before the first Places search: no Google-only restaurant
    PLACES = """(SELECT CAST(NULL AS STRING) AS place_id, CAST(NULL AS STRING) AS place_name,
        CAST(NULL AS STRING) AS restaurant_name_normalized, CAST(NULL AS STRING) AS formatted_address,
        CAST(NULL AS STRING) AS ward, CAST(NULL AS STRING) AS district, CAST(NULL AS STRING) AS city,
        CAST(NULL AS DOUBLE) AS latitude, CAST(NULL AS DOUBLE) AS longitude, CAST(NULL AS STRING) AS primary_type,
        CAST(NULL AS ARRAY<STRING>) AS place_types, CAST(NULL AS DOUBLE) AS rating, CAST(NULL AS INT) AS rating_count,
        CAST(NULL AS STRING) AS price_level, CAST(NULL AS BIGINT) AS price_start_vnd, CAST(NULL AS BIGINT) AS price_end_vnd,
        CAST(NULL AS STRING) AS google_maps_uri, CAST(NULL AS STRING) AS website_uri,
        CAST(NULL AS STRING) AS phone_number, CAST(NULL AS STRING) AS google_opening_hours,
        CAST(NULL AS STRING) AS business_status WHERE false)"""
    print("silver.stg_places_latest does not exist yet: no Google-only restaurants")
if not spark.catalog.tableExists(OSM):  # before the first osm_ingest: no OSM match
    OSM = """(SELECT CAST(NULL AS STRING) AS osm_key, CAST(NULL AS STRING) AS restaurant_name_normalized,
        CAST(NULL AS STRING) AS cuisine, CAST(NULL AS STRING) AS opening_hours, CAST(NULL AS DOUBLE) AS latitude,
        CAST(NULL AS DOUBLE) AS longitude WHERE false)"""
    print("silver.stg_osm_restaurants does not exist yet: no OpenStreetMap match")

# Google place type -> cuisine shown in the dashboard. Order = priority when a place has several types.
CUISINE_TYPES = [
    ("Món Việt", ["vietnamese_restaurant"]),
    ("Món Hoa", ["chinese_restaurant", "cantonese_restaurant", "dim_sum_restaurant", "chinese_noodle_restaurant",
                 "taiwanese_restaurant"]),
    ("Món Nhật", ["japanese_restaurant", "sushi_restaurant", "ramen_restaurant", "japanese_curry_restaurant",
                  "yakiniku_restaurant", "izakaya_restaurant", "tonkatsu_restaurant"]),
    ("Món Hàn", ["korean_restaurant", "korean_barbecue_restaurant"]),
    ("Món Thái", ["thai_restaurant"]),
    ("Món Ấn", ["indian_restaurant", "north_indian_restaurant", "south_indian_restaurant"]),
    ("Món Âu / Mỹ", ["french_restaurant", "italian_restaurant", "pizza_restaurant", "spanish_restaurant",
                     "greek_restaurant", "mediterranean_restaurant", "german_restaurant", "british_restaurant",
                     "european_restaurant", "american_restaurant", "steak_house", "hamburger_restaurant",
                     "mexican_restaurant", "brazilian_restaurant"]),
    ("Món quốc tế khác", ["indonesian_restaurant", "malaysian_restaurant", "filipino_restaurant", "middle_eastern_restaurant",
                          "lebanese_restaurant", "turkish_restaurant", "afghani_restaurant", "african_restaurant"]),
    ("Hải sản", ["seafood_restaurant"]),
    ("Lẩu & nướng", ["hot_pot_restaurant", "barbecue_restaurant"]),
    ("Chay", ["vegan_restaurant", "vegetarian_restaurant"]),
    ("Buffet", ["buffet_restaurant"]),
    ("Ăn nhanh", ["fast_food_restaurant", "sandwich_shop", "snack_bar"]),
    ("Cafe & tráng miệng", ["cafe", "coffee_shop", "tea_house", "dessert_shop", "dessert_restaurant", "bakery",
                            "cake_shop", "ice_cream_shop", "juice_shop", "confectionery"]),
    ("Món Á khác", ["asian_restaurant"]),   # generic: only when no specific type matched
]
# Used when Google has no specific type (most Vietnamese street food is just "restaurant") or only a
# generic one (GENERIC_CUISINES): keywords in the unaccented restaurant name, first match wins
GENERIC_CUISINES = ("Chưa rõ", "Món Á khác")
CUISINE_NAME_KEYWORDS = [
    ("Buffet", r"\\bbuffet\\b"),
    ("Lẩu & nướng", r"\\b(lau|nuong|bbq)\\b"),
    ("Hải sản", r"\\b(hai san|oc)\\b"),
    ("Món Việt", r"\\b(pho|bun|com tam|com|banh mi|banh xeo|banh cuon|banh canh|hu tieu|mi quang|bo kho|chao|xoi|bo ne)\\b"),
    ("Cafe & tráng miệng", r"\\b(cafe|ca phe|coffee|tra sua|che|kem)\\b"),
]
# restaurant_type (2026-11-02): Google primary type first, then words in the unaccented name
RESTAURANT_TYPES = [
    ("Cà phê / trà", ["cafe", "coffee_shop", "tea_house", "juice_shop", "internet_cafe", "cat_cafe", "dog_cafe"]),
    ("Bar / pub", ["bar", "pub", "wine_bar", "cocktail_bar", "lounge_bar", "sports_bar", "brewpub", "beer_garden",
                   "bar_and_grill", "night_club", "karaoke"]),
    ("Tiệm bánh / tráng miệng", ["bakery", "cake_shop", "dessert_shop", "dessert_restaurant", "ice_cream_shop",
                                 "confectionery", "donut_shop", "pastry_shop", "chocolate_shop", "candy_store"]),
    ("Ăn nhanh", ["fast_food_restaurant", "sandwich_shop", "snack_bar", "hamburger_restaurant", "hot_dog_stand"]),
    ("Buffet", ["buffet_restaurant"]),
    ("Khu ẩm thực", ["food_court"]),
]
RESTAURANT_NAME_RX = r"\\b(nha hang|restaurant|bistro|steakhouse)\\b"
EATERY_NAME_RX = r"\\b(quan|xe|tiem|hem|pho|bun|com|banh|hu tieu|mi|oc|lau|chao|xoi|che|bo ne|nuong|an vat)\\b"
spark.createDataFrame(
    [(t, cuisine, prio) for prio, (cuisine, types) in enumerate(CUISINE_TYPES) for t in types],
    "place_type STRING, cuisine STRING, priority INT",
).createOrReplaceTempView("cuisine_types")

# --- GSO (TCTK) administrative units: new wards (after 01/07/2025) and old districts (before 2025) ---
# Built by scripts/build_admin_reference.py from the vietnamadminunits package (MIT, GSO data)
import os
import pandas as pd

def _ref(name):
    path = next(p for p in [f"../../reference/{name}", f"src/reference/{name}"] if os.path.exists(p))
    return pd.read_csv(path, dtype=str, keep_default_na=False)

spark.createDataFrame(_ref("vn_new_ward_to_old_district.csv")).createOrReplaceTempView("ref_ward_raw")
spark.createDataFrame(_ref("vn_old_districts.csv")).createOrReplaceTempView("ref_district_raw")
ADMIN_KEY = "regexp_replace(lower(unaccent_vi(trim({}))), '[^a-z0-9]', '')"   # 'Phường Bến Thành' -> 'phuongbenthanh'
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ref_ward AS
    SELECT province_code AS city_code, province AS city_name,
           regexp_replace(province_key, '^(thanhpho|tinh)', '') AS city_key,
           ward_code, ward AS ward_name, ward_key, ward_short_key,
           CAST(ward_lat AS DOUBLE) AS ward_lat, CAST(ward_lon AS DOUBLE) AS ward_lon,
           nullif(district_old_code, '') AS district_old_code, nullif(districts_old, '') AS districts_old
    FROM ref_ward_raw
""")
spark.sql("""   -- ward name key -> ward, only keys that name one ward in the province
    CREATE OR REPLACE TEMP VIEW ref_ward_key AS
    SELECT city_code, match_key, ward_code
    FROM (SELECT *, COUNT(*) OVER (PARTITION BY city_code, match_key) AS n
          FROM (SELECT DISTINCT w.city_code, k AS match_key, w.ward_code
                FROM ref_ward w LATERAL VIEW explode(array(w.ward_key, w.ward_short_key)) t AS k))
    WHERE n = 1
""")
spark.sql("""   -- district alias key ('quan7', 'q7', 'govap', 'q2' -> Thu Duc) -> old district, unique in the (new) province
    CREATE OR REPLACE TEMP VIEW ref_district_key AS
    SELECT city_code, district_code, district_name, match_key
    FROM (SELECT *, COUNT(*) OVER (PARTITION BY city_code, match_key) AS n
          FROM (SELECT DISTINCT d.new_province_code AS city_code, d.district_code, d.district AS district_name,
                       regexp_replace(a, '[^a-z0-9]', '') AS match_key
                FROM ref_district_raw d LATERAL VIEW explode(split(d.aliases, '[|]')) t AS a))
    WHERE n = 1
""")
print(f"GSO reference: {spark.table('ref_ward').count()} new wards, {spark.table('ref_district_raw').count()} old districts")

run_ts = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]
print(f"Silver source: {SILVER}.stg_restaurant_mentions")
print(f"Gold target: {DIM}, {DISH}, {FACT}")
print(f"🕒 run_ts (valid_from of new versions): {run_ts}")

In [ ]:
# --- dim_restaurant v2 (2026-11-02): keyed by restaurant_id + valid_from, no surrogate key ---
DIM_COLUMNS = """
        restaurant_id              STRING,
        restaurant_name            STRING,
        restaurant_name_normalized STRING,
        restaurant_type            STRING,
        address_full               STRING,
        street                     STRING,
        ward_code                  STRING,
        ward_name                  STRING,
        ward_source                STRING,
        district_code              STRING,
        district_name              STRING,
        district_source            STRING,
        districts_old              STRING,
        city_code                  STRING,
        city_name                  STRING,
        latitude                   DOUBLE,
        longitude                  DOUBLE,
        place_id                   STRING,
        business_status            STRING,
        is_active                  BOOLEAN,
        open_time                  STRING,
        close_time                 STRING,
        is_open_24h                BOOLEAN,
        opening_hours_text         STRING,
        hours_source               STRING,
        cuisine                    STRING,
        cuisine_detail             STRING,
        cuisine_source             STRING,
        price_band                 STRING,
        price_min_vnd              BIGINT,
        price_max_vnd              BIGINT,
        price_source               STRING,
        google_maps_uri            STRING,
        photo_name                 STRING,
        google_website             STRING,
        google_phone               STRING,
        osm_key                    STRING,
        has_youtube_review         BOOLEAN,
        signature_dishes           STRING,
        menu_dishes                STRING,
        video_price_min_vnd        BIGINT,
        video_price_max_vnd        BIGINT,
        valid_from                 TIMESTAMP,
        valid_to                   TIMESTAMP,
        is_current                 BOOLEAN,
        tracked_checksum           STRING,
        etl_date                   TIMESTAMP"""
DIM_COMMENT = ("Restaurant dimension v2 (SCD 2 on address codes / location, the rest type 1). Key: restaurant_id + valid_from "
               "(no surrogate key); restaurant_id unknown = the Unknown row. GSO (TCTK) ward / old district / province codes.")

# --- One-time migration from v1 (restaurant_key, address / ward / district / city text): keep every version ---
MIGRATED = False
if spark.catalog.tableExists(DIM) and "restaurant_key" in spark.table(DIM).columns:
    old_cols = set(spark.table(DIM).columns)
    v1_version = spark.sql(f"DESCRIBE HISTORY {DIM} LIMIT 1").collect()[0]["version"]   # time travel source
    def old(col, cast="STRING"):
        return col if col in old_cols else f"CAST(NULL AS {cast})"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {DIM} ({DIM_COLUMNS}
        ) USING DELTA COMMENT '{DIM_COMMENT}'
    """)
    spark.sql(f"""
        INSERT INTO {DIM} (restaurant_id, restaurant_name, restaurant_name_normalized, address_full, ward_name, district_name,
                           city_name, latitude, longitude, place_id, cuisine, cuisine_detail, cuisine_source, price_band,
                           price_min_vnd, price_max_vnd, price_source, google_maps_uri, photo_name, google_website, google_phone,
                           osm_key, has_youtube_review, signature_dishes, menu_dishes, video_price_min_vnd, video_price_max_vnd,
                           opening_hours_text, valid_from, valid_to, is_current, tracked_checksum, etl_date)
        SELECT CASE WHEN restaurant_key = -1 THEN 'unknown'
                    WHEN restaurant_id LIKE 'place:%' THEN md5(restaurant_id) ELSE restaurant_id END, restaurant_name, restaurant_name_normalized,
               address, ward, district, city, latitude, longitude, place_id, {old('cuisine')}, {old('cuisine_detail')},
               {old('cuisine_source')}, {old('price_band')}, {old('price_min_vnd', 'BIGINT')}, {old('price_max_vnd', 'BIGINT')},
               {old('price_source')}, {old('google_maps_uri')}, {old('photo_name')}, {old('google_website')},
               {old('google_phone')}, {old('osm_key')}, {old('has_youtube_review', 'BOOLEAN')}, {old('signature_dishes')},
               {old('menu_dishes')}, {old('video_price_min_vnd', 'BIGINT')}, {old('video_price_max_vnd', 'BIGINT')},
               COALESCE({old('google_opening_hours')}, {old('video_opening_hours')}, {old('opening_hours')}),
               valid_from, valid_to, is_current, tracked_checksum, etl_date
        FROM {DIM} VERSION AS OF {v1_version}
    """)
    MIGRATED = True
    print(f"🔁 {DIM}: migrated to v2 (restaurant_key dropped, -1 -> 'unknown'); current versions get the new address codes "
          "below without opening a new version")
spark.sql(f"CREATE TABLE IF NOT EXISTS {DIM} ({DIM_COLUMNS}\n) USING DELTA COMMENT '{DIM_COMMENT}'")

# --- Every mention with its durable restaurant_id (dedup rule above; NULL until geocode matched it) ---
spark.sql(rf"""
    CREATE OR REPLACE TEMP VIEW mentions_keyed AS
    WITH m AS (
        SELECT *,
            nullif(trim(regexp_replace(regexp_replace(regexp_replace(
                lower(unaccent_vi(coalesce(district, ''))),
                '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
                '^(tp\\.?|thanh pho)\\s+', ''),
                '\\s+', ' ')), '') AS district_norm
        FROM {SILVER}.stg_restaurant_mentions
    )
    SELECT m.*,
           g.place_id, g.place_name, g.formatted_address, g.ward AS geo_ward, g.city AS geo_city,
           g.latitude, g.longitude,
           -- durable id from the Google place only (same place = same restaurant); NULL = not matched yet
           md5('place:' || g.place_id) AS restaurant_id
    FROM m
    LEFT JOIN {GEOCODE} g
      ON g.restaurant_name_normalized = m.restaurant_name_normalized AND g.district_norm <=> m.district_norm
     AND g.match_status = 'matched'
""")

# --- One row per restaurant_id: latest mention wins (prefer rows with an address) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_src AS
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY restaurant_id ORDER BY place_id IS NULL, address IS NULL, extracted_at DESC, video_id) AS rn
        FROM mentions_keyed
        WHERE restaurant_id IS NOT NULL   -- not geocoded yet: not in the dimension (fact: Unknown)
    )
    SELECT restaurant_id,
           COALESCE(place_name, restaurant_name)    AS restaurant_name,
           restaurant_name_normalized,
           COALESCE(formatted_address, address)     AS address,
           geo_ward,                                -- Google's (new) ward, parsed from the formatted address
           COALESCE(geo_city, city)                 AS city,
           district                                 AS video_district,   -- old district as said in the video
           latitude, longitude, place_id,
           true AS has_youtube_review
    FROM ranked WHERE rn = 1
    UNION ALL
    -- Google-only: places found by Text Search that no video mentions (same md5('place:' || place_id) id).
    -- Their district is the search grid's keyword, not a fact about the place: not used.
    SELECT md5('place:' || p.place_id), p.place_name, p.restaurant_name_normalized, p.formatted_address, p.ward,
           p.city, CAST(NULL AS STRING), p.latitude, p.longitude, p.place_id, false
    FROM {PLACES} p
    WHERE p.place_id IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM mentions_keyed m WHERE m.restaurant_id = md5('place:' || p.place_id))
""")
spark.sql("""
    CREATE OR REPLACE TEMP VIEW mention_restaurant AS
    SELECT DISTINCT video_id, restaurant_id, mention_source,
           concat(restaurant_name_normalized, '|', coalesce(district_norm, '')) AS name_key
    FROM mentions_keyed
""")
# --- Type 1 attributes from Google Place Details (one row per restaurant_id) ---
# Place Details columns added 2026-11-02 (stg/restaurant_place_details adds them; NULL until a place is re-read)
details_cols = set(spark.table(DETAILS).columns)
def dcol(c):
    return f"p.{c}" if c in details_cols else "CAST(NULL AS STRING)"
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_details AS
    WITH d AS (   -- Place Details when fetched, else the same Enterprise fields from the Text Search result
        SELECT s.restaurant_id, p.primary_type, p.primary_type_display_name, p.place_types, p.rating, p.rating_count,
               p.price_level, p.price_start_vnd, p.price_end_vnd, p.google_maps_uri, p.photo_name,
               p.website_uri, p.phone_number, p.google_opening_hours,
               {dcol('business_status')} AS business_status, {dcol('opening_periods')} AS opening_periods,
               {dcol('google_ward')} AS google_ward, {dcol('google_district')} AS google_district,
               {dcol('google_city')} AS google_city
        FROM restaurant_src s
        JOIN {DETAILS} p ON p.place_id = s.place_id AND p.fetch_status = 'ok'
        UNION ALL
        SELECT s.restaurant_id, p.primary_type, CAST(NULL AS STRING), p.place_types, p.rating, p.rating_count,
               p.price_level, p.price_start_vnd, p.price_end_vnd, p.google_maps_uri, CAST(NULL AS STRING),
               p.website_uri, p.phone_number, p.google_opening_hours,
               p.business_status, CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING)
        FROM restaurant_src s
        JOIN {PLACES} p ON p.place_id = s.place_id
        LEFT ANTI JOIN {DETAILS} x ON x.place_id = s.place_id AND x.fetch_status = 'ok'
    ),
    typed AS (   -- primary type first, then the other Google types in their order
        SELECT d.restaurant_id, t.place_type, -1 AS pos FROM d LATERAL VIEW explode(array(d.primary_type)) t AS place_type
        UNION ALL
        SELECT d.restaurant_id, t.place_type, t.pos FROM d LATERAL VIEW posexplode(d.place_types) t AS pos, place_type
    ),
    cuisine AS (   -- most specific cuisine first (CUISINE_TYPES order), then Google's type order
        SELECT restaurant_id, min_by(c.cuisine, struct(c.priority, t.pos)) AS cuisine
        FROM typed t JOIN cuisine_types c ON c.place_type = t.place_type
        GROUP BY restaurant_id
    ),
    priced AS (
        SELECT d.*, COALESCE((price_start_vnd + price_end_vnd) / 2, price_start_vnd) AS price_mid_vnd FROM d
    )
    SELECT p.restaurant_id,
           COALESCE(c.cuisine, 'Chưa rõ')                       AS google_cuisine,
           p.primary_type_display_name                          AS cuisine_detail,
           CASE
               WHEN p.price_mid_vnd <  100000 THEN '1 · Dưới 100k'
               WHEN p.price_mid_vnd <  300000 THEN '2 · 100k–300k'
               WHEN p.price_mid_vnd <  600000 THEN '3 · 300k–600k'
               WHEN p.price_mid_vnd >= 600000 THEN '4 · Trên 600k'
               WHEN p.price_level IN ('PRICE_LEVEL_FREE', 'PRICE_LEVEL_INEXPENSIVE') THEN '1 · Dưới 100k'
               WHEN p.price_level = 'PRICE_LEVEL_MODERATE'       THEN '2 · 100k–300k'
               WHEN p.price_level = 'PRICE_LEVEL_EXPENSIVE'      THEN '3 · 300k–600k'
               WHEN p.price_level = 'PRICE_LEVEL_VERY_EXPENSIVE' THEN '4 · Trên 600k'
               ELSE 'Chưa rõ'
           END                                                  AS price_band,
           p.price_start_vnd                                    AS price_min_vnd,
           p.price_end_vnd                                      AS price_max_vnd,
           p.google_maps_uri,
           p.photo_name,
           p.website_uri                                        AS google_website,
           p.phone_number                                       AS google_phone,
           p.google_opening_hours,
           p.primary_type,
           p.business_status, p.opening_periods, p.google_ward, p.google_district, p.google_city
    FROM priced p
    LEFT JOIN cuisine c ON c.restaurant_id = p.restaurant_id
""")
# --- OpenStreetMap match: nearest OSM place within OSM_MAX_DISTANCE_M with a similar name ---
deg = OSM_MAX_DISTANCE_M / 111000 * 1.5   # bounding box (degrees) before the exact distance
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_osm AS
    WITH c AS (
        SELECT s.restaurant_id, o.osm_key, o.cuisine AS osm_cuisine, o.opening_hours,
               s.restaurant_name_normalized AS a, o.restaurant_name_normalized AS b,
               2 * 6371000 * asin(sqrt(
                   pow(sin(radians(o.latitude - s.latitude) / 2), 2)
                   + cos(radians(s.latitude)) * cos(radians(o.latitude)) * pow(sin(radians(o.longitude - s.longitude) / 2), 2)
               )) AS distance_m
        FROM restaurant_src s
        JOIN {OSM} o
          ON o.latitude BETWEEN s.latitude - {deg} AND s.latitude + {deg}
         AND o.longitude BETWEEN s.longitude - {deg} AND s.longitude + {deg}
        WHERE s.latitude IS NOT NULL AND s.longitude IS NOT NULL
          AND COALESCE(s.restaurant_name_normalized, '') <> '' AND COALESCE(o.restaurant_name_normalized, '') <> ''
    ),
    ok AS (
        SELECT * FROM c
        WHERE distance_m <= {OSM_MAX_DISTANCE_M}
          AND (a = b
               OR (length(b) >= 4 AND instr(a, b) > 0)
               OR (length(a) >= 4 AND instr(b, a) > 0)
               OR levenshtein(a, b) <= greatest(2, CAST(0.25 * greatest(length(a), length(b)) AS INT)))
    )
    SELECT restaurant_id,
           min_by(osm_key, struct(distance_m, osm_key))       AS osm_key,
           min_by(osm_cuisine, struct(distance_m, osm_key))   AS osm_cuisine,
           min_by(opening_hours, struct(distance_m, osm_key)) AS opening_hours
    FROM ok GROUP BY restaurant_id
""")

# --- What the reviews say (stg_restaurant_mentions, EXTRACTION_VERSION 2): dishes, prices, opening hours ---
has_dishes = "dishes" in spark.table(f"{SILVER}.stg_restaurant_mentions").columns
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_video_menu AS
    WITH d AS (
        SELECT m.restaurant_id, trim(x.name) AS dish, lower(unaccent_vi(trim(x.name))) AS dish_key,
               CASE WHEN x.price_vnd BETWEEN 5000 AND 5000000 THEN x.price_vnd END AS price_vnd,
               coalesce(x.is_signature, false) AS is_signature, m.video_id
        FROM mentions_keyed m
        LATERAL VIEW explode({'m.dishes' if has_dishes else 'CAST(NULL AS ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>)'}) t AS x
        WHERE m.restaurant_id IS NOT NULL AND length(trim(x.name)) > 1
    ),
    per_dish AS (
        SELECT restaurant_id, dish_key, max_by(dish, length(dish)) AS dish,
               COUNT(DISTINCT video_id) AS videos, COUNT_IF(is_signature) AS signature_votes
        FROM d GROUP BY restaurant_id, dish_key
    ),
    menu AS (
        SELECT restaurant_id,
               array_join(slice(transform(filter(array_sort(collect_list(struct(-signature_votes AS s, -videos AS v, dish))),
                                                 x -> x.s < 0), x -> x.dish), 1, 5), ', ') AS signature_dishes,
               array_join(slice(transform(array_sort(collect_list(struct(-videos AS v, -signature_votes AS s, dish))),
                                          x -> x.dish), 1, 10), ', ') AS menu_dishes
        FROM per_dish GROUP BY restaurant_id
    ),
    price AS (
        SELECT restaurant_id, MIN(price_vnd) AS video_price_min_vnd, MAX(price_vnd) AS video_price_max_vnd,
               percentile_approx(price_vnd, 0.5) AS video_price_median_vnd
        FROM d WHERE price_vnd IS NOT NULL GROUP BY restaurant_id
    ),
    hours AS (
        SELECT restaurant_id, max_by({'opening_hours' if has_dishes else 'CAST(NULL AS STRING)'}, extracted_at) AS video_opening_hours
        FROM mentions_keyed
        WHERE restaurant_id IS NOT NULL AND {'opening_hours IS NOT NULL' if has_dishes else 'false'}
        GROUP BY restaurant_id
    )
    SELECT k.restaurant_id, nullif(m.signature_dishes, '') AS signature_dishes, nullif(m.menu_dishes, '') AS menu_dishes,
           h.video_opening_hours, p.video_price_min_vnd, p.video_price_max_vnd, p.video_price_median_vnd
    FROM (SELECT DISTINCT restaurant_id FROM mentions_keyed WHERE restaurant_id IS NOT NULL) k
    LEFT JOIN menu m  ON m.restaurant_id = k.restaurant_id
    LEFT JOIN price p ON p.restaurant_id = k.restaurant_id
    LEFT JOIN hours h ON h.restaurant_id = k.restaurant_id
""")

# --- Address with GSO codes: province -> new ward -> old district (rules in the header) ---
K = ADMIN_KEY.format
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_location AS
    WITH s AS (
        SELECT s.restaurant_id, s.address, s.latitude, s.longitude, s.place_id, s.video_district,
               COALESCE(d.google_ward, s.geo_ward)  AS g_ward,
               d.google_district                    AS g_district,
               COALESCE(d.google_city, s.city)      AS g_city
        FROM restaurant_src s LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
    ),
    city AS (      -- province from Google's city name ('Hồ Chí Minh', 'Thành phố Hà Nội')
        SELECT s.*, c.city_code
        FROM s LEFT JOIN (SELECT DISTINCT city_code, city_key FROM ref_ward) c
          ON c.city_key = regexp_replace({K('s.g_city')}, '^(thanhpho|tinh|tp)', '')
    ),
    by_name AS (   -- new ward from Google's ward name, inside the province
        SELECT c.*, k.ward_code AS named_ward_code
        FROM city c LEFT JOIN ref_ward_key k ON k.city_code = c.city_code AND k.match_key = {K('c.g_ward')}
    ),
    nearest AS (   -- else the nearest new-ward centroid (in the province when known)
        SELECT b.restaurant_id,
               min_by(w.ward_code, pow(w.ward_lat - b.latitude, 2)
                                   + pow((w.ward_lon - b.longitude) * cos(radians(b.latitude)), 2)) AS ward_code
        FROM by_name b JOIN ref_ward w
          ON (b.city_code IS NULL OR w.city_code = b.city_code)
         AND w.ward_lat BETWEEN b.latitude - 0.5 AND b.latitude + 0.5
         AND w.ward_lon BETWEEN b.longitude - 0.5 AND b.longitude + 0.5
        WHERE b.named_ward_code IS NULL AND b.latitude IS NOT NULL AND b.longitude IS NOT NULL
        GROUP BY b.restaurant_id
    ),
    ward AS (
        SELECT b.*, COALESCE(b.named_ward_code, n.ward_code) AS ward_code,
               CASE WHEN b.named_ward_code IS NOT NULL THEN 'google' WHEN n.ward_code IS NOT NULL THEN 'nearest' END AS ward_source
        FROM by_name b LEFT JOIN nearest n ON n.restaurant_id = b.restaurant_id
    ),
    w AS (
        SELECT x.*, r.ward_name, r.district_old_code, r.districts_old,
               COALESCE(x.city_code, r.city_code) AS city_code_final, r.city_name AS ward_city_name
        FROM ward x LEFT JOIN ref_ward r ON r.ward_code = x.ward_code
    ),
    district AS (  -- old district: Google addressComponents > video > the ward's only old district
        SELECT w.*,
               gd.district_code AS google_district_code,
               COALESCE(vd.district_code, vd2.district_code) AS video_district_code
        FROM w
        LEFT JOIN ref_district_key gd ON gd.city_code = w.city_code_final AND gd.match_key = {K('w.g_district')}
        LEFT JOIN ref_district_key vd ON vd.city_code = w.city_code_final
             AND vd.match_key = regexp_replace({K('w.video_district')}, '(?<=[a-z])0+(?=[0-9])', '')
        LEFT JOIN ref_district_key vd2 ON vd2.city_code = w.city_code_final
             AND vd2.match_key = regexp_replace(regexp_replace({K('w.video_district')}, '(?<=[a-z])0+(?=[0-9])', ''),
                                                '^(tp|thanhpho|tx|thixa|huyen)', '')
    ),
    final AS (
        SELECT d.*,
               COALESCE(d.google_district_code, d.video_district_code, d.district_old_code) AS district_code,
               CASE WHEN d.google_district_code IS NOT NULL THEN 'google'
                    WHEN d.video_district_code IS NOT NULL THEN 'video'
                    WHEN d.district_old_code IS NOT NULL THEN 'ward' END AS district_source
        FROM district d
    )
    SELECT f.restaurant_id,
           f.address                                                AS address_full,
           -- street = the part before ", <ward>, ..." (else everything but the last two parts)
           nullif(trim(CASE
               WHEN f.g_ward IS NOT NULL AND f.address RLIKE concat(',\\\\s*\\\\Q', f.g_ward, '\\\\E\\\\s*(,|$)')
                   THEN regexp_replace(f.address, concat(',\\\\s*\\\\Q', f.g_ward, '\\\\E\\\\s*(,.*)?$'), '')
               WHEN size(split(f.address, ',')) >= 3
                   THEN array_join(slice(split(f.address, ','), 1, size(split(f.address, ',')) - 2), ',')
           END), '')                                                AS street,
           f.ward_code, f.ward_name, f.ward_source,
           f.district_code, rd.district_name, f.district_source, f.districts_old,
           f.city_code_final                                        AS city_code,
           COALESCE(f.ward_city_name, rc.city_name)                 AS city_name,
           f.latitude, f.longitude, f.place_id
    FROM final f
    LEFT JOIN (SELECT DISTINCT district_code, district_name FROM ref_district_key) rd ON rd.district_code = f.district_code
    LEFT JOIN (SELECT DISTINCT city_code, city_name FROM ref_ward) rc ON rc.city_code = f.city_code_final
""")
loc = spark.sql("""
    SELECT COUNT(*) AS n, COUNT(city_code) AS city, COUNT(ward_code) AS ward, COUNT_IF(ward_source = 'nearest') AS ward_nearest,
           COUNT(district_code) AS district, COUNT_IF(district_source = 'google') AS d_google,
           COUNT_IF(district_source = 'video') AS d_video, COUNT_IF(district_source = 'ward') AS d_ward
    FROM restaurant_location""").collect()[0]
print(f"Address codes: {loc['n']} restaurants · province {loc['city']} · new ward {loc['ward']} "
      f"({loc['ward_nearest']} by nearest centroid) · old district {loc['district']} "
      f"(google {loc['d_google']}, video {loc['d_video']}, ward {loc['d_ward']})")

# --- Opening hours: Google periods > hours said in the video > OpenStreetMap ---
PERIODS = "ARRAY<STRUCT<open: STRUCT<day: INT, hour: INT, minute: INT>, close: STRUCT<day: INT, hour: INT, minute: INT>>>"
def hhmm(m):
    return f"concat(lpad(CAST(floor(({m}) / 60) % 24 AS STRING), 2, '0'), ':', lpad(CAST(({m}) % 60 AS STRING), 2, '0'))"
# free text ('6h - 22h', '06:00-22:00', 'Mo-Su 07:00-21:30', '6h30 đến 10h tối'): first time = open, last = close
TIME_RX = "([0-9]{1,2})\\\\s*(?:h|g|giờ|:)\\\\s*([0-9]{2})?"
def text_minutes(col, idx):
    return f"""CASE WHEN size(regexp_extract_all({col}, '{TIME_RX}', 1)) >= 2 THEN
        CAST(element_at(regexp_extract_all({col}, '{TIME_RX}', 1), {idx}) AS INT) * 60
        + COALESCE(TRY_CAST(element_at(regexp_extract_all({col}, '{TIME_RX}', 2), {idx}) AS INT), 0) END"""
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_hours_text AS
    SELECT restaurant_id, src, txt,
           {text_minutes('txt', 1)} AS open_m,
           {text_minutes('txt', -1)} AS close_m0,
           txt RLIKE '(?i)(24\\\\s*/\\\\s*24|24/7|24h|cả ngày|suốt ngày|00:00\\\\s*-\\\\s*24:00)' AS is_24h
    FROM (SELECT restaurant_id, 'video' AS src, video_opening_hours AS txt FROM restaurant_video_menu WHERE video_opening_hours IS NOT NULL
          UNION ALL
          SELECT restaurant_id, 'osm', opening_hours FROM restaurant_osm WHERE opening_hours IS NOT NULL)
""")
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_hours AS
    WITH g AS (
        SELECT restaurant_id, google_opening_hours, from_json(opening_periods, '{PERIODS}') AS p
        FROM restaurant_details
    ),
    gm AS (
        SELECT restaurant_id, google_opening_hours,
               size(p) = 1 AND p[0].close IS NULL AS is_24h,   -- Google's "open 24/7": one period, no close
               array_min(transform(p, x -> x.open.hour * 60 + x.open.minute)) AS open_m,
               array_max(transform(filter(p, x -> x.close IS NOT NULL),
                                   x -> ((x.close.day - x.open.day + 7) % 7) * 1440 + x.close.hour * 60 + x.close.minute)) AS close_m
        FROM g WHERE p IS NOT NULL AND size(p) > 0
    ),
    t AS (   -- '10h tối' / '9pm': an evening closing below 12h means + 12h
        SELECT restaurant_id, src, txt, is_24h, open_m,
               CASE WHEN close_m0 < 720 AND (close_m0 <= open_m OR txt RLIKE '(?i)(tối|chiều|đêm|pm)') THEN close_m0 + 720
                    ELSE close_m0 END AS close_m
        FROM restaurant_hours_text
        WHERE is_24h OR (open_m < 1440 AND close_m0 <= 1440)
    ),
    tv AS (SELECT * FROM t WHERE src = 'video'),
    tos AS (SELECT * FROM t WHERE src = 'osm')
    SELECT s.restaurant_id,
           CASE WHEN gm.restaurant_id IS NOT NULL THEN 'google' WHEN tv.restaurant_id IS NOT NULL THEN 'video'
                WHEN tos.restaurant_id IS NOT NULL THEN 'osm' END                                     AS hours_source,
           COALESCE(gm.is_24h, tv.is_24h, tos.is_24h)                                                 AS is_open_24h,
           CASE WHEN COALESCE(gm.is_24h, tv.is_24h, tos.is_24h) THEN '00:00'
                ELSE {hhmm('COALESCE(gm.open_m, tv.open_m, tos.open_m)')} END                          AS open_time,
           CASE WHEN COALESCE(gm.is_24h, tv.is_24h, tos.is_24h) THEN '24:00'
                ELSE {hhmm('COALESCE(gm.close_m, tv.close_m, tos.close_m)')} END                       AS close_time,
           COALESCE(d.google_opening_hours, vm.video_opening_hours, o.opening_hours)                AS opening_hours_text
    FROM restaurant_src s
    LEFT JOIN gm ON gm.restaurant_id = s.restaurant_id
    LEFT JOIN tv ON tv.restaurant_id = s.restaurant_id
    LEFT JOIN tos ON tos.restaurant_id = s.restaurant_id
    LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
    LEFT JOIN restaurant_video_menu vm ON vm.restaurant_id = s.restaurant_id
    LEFT JOIN restaurant_osm o ON o.restaurant_id = s.restaurant_id
""")

# --- Final cuisine: Google specific type > keyword in the name > OSM cuisine > Google generic > Chưa rõ ---
name_rule = " ".join(f"WHEN lower(unaccent_vi(s.restaurant_name)) RLIKE '{rx}' THEN '{cuisine}'"
                     for cuisine, rx in CUISINE_NAME_KEYWORDS)
type_rule = " ".join(f"WHEN d.primary_type IN ({', '.join(repr(t) for t in types)}) THEN '{label}'"
                     for label, types in RESTAURANT_TYPES)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_attributes AS
    WITH a AS (
        SELECT s.restaurant_id, s.restaurant_name, s.has_youtube_review,
               d.google_cuisine, d.cuisine_detail, d.price_band, d.price_min_vnd, d.price_max_vnd,
               d.google_maps_uri, d.photo_name,
               d.google_website, d.google_phone, d.google_opening_hours,
               CASE {name_rule} END AS name_cuisine,
               o.osm_cuisine, o.osm_key,
               vm.signature_dishes, vm.menu_dishes,
               vm.video_price_min_vnd, vm.video_price_max_vnd, vm.video_price_median_vnd,
               CASE {type_rule} END AS google_type, d.primary_type,
               lower(unaccent_vi(s.restaurant_name)) AS name_plain,
               d.business_status,
               h.open_time, h.close_time, h.is_open_24h, h.opening_hours_text, h.hours_source
        FROM restaurant_src s
        LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_osm o ON o.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_video_menu vm ON vm.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_hours h ON h.restaurant_id = s.restaurant_id
    ),
    b AS (
        SELECT a.*,
           -- Google price first; else the median dish price stated in the reviews (2026-11-02)
           COALESCE(price_band,
                    CASE WHEN video_price_median_vnd <  100000 THEN '1 · Dưới 100k'
                         WHEN video_price_median_vnd <  300000 THEN '2 · 100k–300k'
                         WHEN video_price_median_vnd <  600000 THEN '3 · 300k–600k'
                         WHEN video_price_median_vnd >= 600000 THEN '4 · Trên 600k' END,
                    'Chưa rõ') AS final_price_band
        FROM a
    )
    SELECT restaurant_id, restaurant_name,
           CASE WHEN google_type IS NOT NULL THEN google_type
                WHEN name_plain RLIKE '{RESTAURANT_NAME_RX}' THEN 'Nhà hàng'
                WHEN name_plain RLIKE '{EATERY_NAME_RX}' THEN 'Quán ăn'
                WHEN primary_type LIKE '%restaurant%' AND left(final_price_band, 1) IN ('3', '4') THEN 'Nhà hàng'
                WHEN primary_type LIKE '%restaurant%' OR primary_type IN ('meal_takeaway', 'meal_delivery', 'food') THEN 'Quán ăn'
                ELSE 'Chưa rõ' END AS restaurant_type,
           business_status,
           NOT COALESCE(business_status = 'CLOSED_PERMANENTLY', false) AS is_active,
           open_time, close_time, is_open_24h, opening_hours_text, hours_source,
           CASE WHEN COALESCE(google_cuisine, 'Chưa rõ') NOT IN {GENERIC_CUISINES} THEN google_cuisine
                WHEN name_cuisine IS NOT NULL THEN name_cuisine
                WHEN osm_cuisine IS NOT NULL THEN osm_cuisine
                ELSE COALESCE(google_cuisine, 'Chưa rõ') END AS cuisine,
           CASE WHEN COALESCE(google_cuisine, 'Chưa rõ') NOT IN {GENERIC_CUISINES} THEN 'google'
                WHEN name_cuisine IS NOT NULL THEN 'name'
                WHEN osm_cuisine IS NOT NULL THEN 'osm'
                WHEN COALESCE(google_cuisine, 'Chưa rõ') <> 'Chưa rõ' THEN 'google' END AS cuisine_source,
           cuisine_detail,
           final_price_band AS price_band,
           CASE WHEN price_band IS NOT NULL OR price_min_vnd IS NOT NULL THEN 'google'
                WHEN video_price_median_vnd IS NOT NULL THEN 'video' END AS price_source,
           CASE WHEN price_band IS NOT NULL OR price_min_vnd IS NOT NULL THEN price_min_vnd ELSE video_price_min_vnd END AS price_min_vnd,
           CASE WHEN price_band IS NOT NULL OR price_min_vnd IS NOT NULL THEN price_max_vnd ELSE video_price_max_vnd END AS price_max_vnd,
           google_maps_uri, photo_name, has_youtube_review,
           osm_key, google_website, google_phone,
           signature_dishes, menu_dishes, video_price_min_vnd, video_price_max_vnd
    FROM b
""")
print(f"OpenStreetMap matches: {spark.table('restaurant_osm').count()}")
print(f"Restaurants in source: {spark.table('restaurant_src').count()} "
      f"(Google-only: {spark.table('restaurant_src').where('NOT has_youtube_review').count()}) · "
      f"with Google details: {spark.table('restaurant_details').count()}")
display(spark.sql("SELECT cuisine, COUNT(*) AS n FROM restaurant_attributes GROUP BY 1 ORDER BY 2 DESC"))

In [ ]:
# --- SCD 2 source: tracked location (codes) + checksum ---
TRACKED = ["address_full", "street", "ward_code", "ward_name", "ward_source", "district_code", "district_name",
           "district_source", "districts_old", "city_code", "city_name", "latitude", "longitude", "place_id"]
spark.sql("""
    CREATE OR REPLACE TEMP VIEW restaurant_scd2_src AS
    SELECT s.restaurant_id, s.restaurant_name, s.restaurant_name_normalized,
           l.address_full, l.street, l.ward_code, l.ward_name, l.ward_source, l.district_code, l.district_name,
           l.district_source, l.districts_old, l.city_code, l.city_name, l.latitude, l.longitude, l.place_id,
           md5(concat_ws('||', l.address_full, l.ward_code, l.district_code, l.city_code,
                         CAST(l.latitude AS STRING), CAST(l.longitude AS STRING), l.place_id)) AS tracked_checksum
    FROM restaurant_src s JOIN restaurant_location l ON l.restaurant_id = s.restaurant_id
""")

# One-time (migration run): the current v1 versions get the v2 address columns and checksum in place,
# so the new checksum definition does not open a new version for every restaurant
if MIGRATED:
    spark.sql(f"""
        MERGE INTO {DIM} AS t
        USING restaurant_scd2_src AS s ON t.restaurant_id = s.restaurant_id AND t.is_current
        WHEN MATCHED THEN UPDATE SET {', '.join(f't.{c} = s.{c}' for c in TRACKED)},
            t.tracked_checksum = s.tracked_checksum, t.etl_date = current_timestamp()
    """)

# --- SCD 2 MERGE: close the changed current version and insert its new version in one atomic MERGE ---
spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING (
        SELECT s.restaurant_id AS merge_key, s.* FROM restaurant_scd2_src s
        UNION ALL
        -- changed restaurants appear a second time with a NULL key, so they also hit WHEN NOT MATCHED (new version)
        SELECT NULL AS merge_key, s.*
        FROM restaurant_scd2_src s
        JOIN {DIM} d ON d.restaurant_id = s.restaurant_id AND d.is_current AND d.tracked_checksum <> s.tracked_checksum
    ) AS s
    ON t.restaurant_id = s.merge_key AND t.is_current
    WHEN MATCHED AND t.tracked_checksum <> s.tracked_checksum THEN
        UPDATE SET t.valid_to = TIMESTAMP'{run_ts}', t.is_current = false, t.etl_date = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
        restaurant_id, restaurant_name, restaurant_name_normalized, {', '.join(TRACKED)},
        valid_from, valid_to, is_current, tracked_checksum, etl_date
    ) VALUES (
        s.restaurant_id, s.restaurant_name, s.restaurant_name_normalized, {', '.join(f's.{c}' for c in TRACKED)},
        TIMESTAMP'{run_ts}', TIMESTAMP'9999-12-31 00:00:00', true, s.tracked_checksum, current_timestamp()
    )
    -- a restaurant that left the source: close it
    WHEN NOT MATCHED BY SOURCE AND t.is_current AND t.restaurant_id <> 'unknown' THEN
        UPDATE SET t.valid_to = TIMESTAMP'{run_ts}', t.is_current = false, t.etl_date = current_timestamp()
""")

# --- Type 1: name, type, status, hours, Google details, video dishes overwrite every version ---
TYPE1 = ["restaurant_name", "restaurant_type", "business_status", "is_active", "open_time", "close_time", "is_open_24h",
         "opening_hours_text", "hours_source", "cuisine", "cuisine_detail", "cuisine_source", "price_band",
         "price_min_vnd", "price_max_vnd", "price_source", "google_maps_uri", "photo_name", "google_website",
         "google_phone", "osm_key", "has_youtube_review", "signature_dishes", "menu_dishes",
         "video_price_min_vnd", "video_price_max_vnd"]
spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING restaurant_attributes AS s ON t.restaurant_id = s.restaurant_id
    WHEN MATCHED AND NOT ({' AND '.join(f't.{c} <=> s.{c}' for c in TYPE1)}) THEN
        UPDATE SET {', '.join(f't.{c} = s.{c}' for c in TYPE1)}, t.etl_date = current_timestamp()
""")
spark.sql(f"""
    INSERT INTO {DIM} (restaurant_id, restaurant_name, restaurant_type, is_active, valid_from, valid_to, is_current, etl_date)
    SELECT 'unknown', 'Unknown', 'Chưa rõ', true, TIMESTAMP'1900-01-01 00:00:00', TIMESTAMP'9999-12-31 00:00:00', true,
           current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {DIM} WHERE restaurant_id = 'unknown')
""")
v = spark.sql(f"SELECT COUNT(*) AS n, COUNT_IF(is_current) AS cur FROM {DIM} WHERE restaurant_id <> 'unknown'").collect()[0]
print(f"✅ {DIM}: {v['n']} versions, {v['cur']} current")
display(spark.sql(f"""
    SELECT restaurant_type, COUNT(*) AS restaurants, COUNT_IF(NOT is_active) AS closed, COUNT(business_status) AS with_status,
           COUNT(open_time) AS with_hours, COUNT(ward_code) AS with_ward, COUNT(district_code) AS with_district
    FROM {DIM} WHERE is_current AND restaurant_id <> 'unknown' GROUP BY 1 ORDER BY 2 DESC
"""))
display(spark.sql(f"""
    SELECT city_name, district_name, district_source, COUNT(*) AS restaurants
    FROM {DIM} WHERE is_current AND restaurant_id <> 'unknown' GROUP BY ALL ORDER BY restaurants DESC LIMIT 40
"""))
display(spark.sql(f"""
    SELECT hours_source, price_source, cuisine_source, COUNT(*) AS restaurants
    FROM {DIM} WHERE is_current AND restaurant_id <> 'unknown' GROUP BY ALL ORDER BY restaurants DESC
"""))

In [ ]:
# --- fact_video_restaurant_mention: video_key x restaurant_id (v2: no restaurant_key) ---
if spark.catalog.tableExists(FACT) and "restaurant_key" in spark.table(FACT).columns:
    spark.sql(f"DROP TABLE {FACT}")   # v1 (restaurant_key): rebuilt in full below from Silver
    print(f"🔁 {FACT}: v1 (restaurant_key) dropped, rebuilt with restaurant_id")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT} (
        video_key      BIGINT,
        restaurant_id  STRING,
        mention_source STRING,
        mention_count  TINYINT,
        mention_weight DOUBLE,
        etl_date       TIMESTAMP
    )
    USING DELTA
    COMMENT 'Factless bridge: 1 video x 1 restaurant (restaurant_id, unknown = not geocoded yet). mention_weight = 1 / restaurants in the video (sums to 1).'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW fact_mention_src AS
    WITH m AS (
        -- unmatched mentions (restaurant_id NULL) collapse into one Unknown row per video that keeps their share
        SELECT video_id, COALESCE(restaurant_id, 'unknown') AS restaurant_id, max(mention_source) AS mention_source,
               CASE WHEN restaurant_id IS NULL THEN COUNT(DISTINCT name_key) ELSE 1 END AS n
        FROM mention_restaurant GROUP BY video_id, restaurant_id
    )
    SELECT
        COALESCE(dv.video_key, -1)                              AS video_key,
        m.restaurant_id,
        m.mention_source,
        CAST(least(m.n, 127) AS TINYINT)                        AS mention_count,
        1.0D * m.n / SUM(m.n) OVER (PARTITION BY m.video_id)    AS mention_weight,
        current_timestamp()                                     AS etl_date
    FROM m
    LEFT JOIN {GOLD}.dim_video dv ON dv.video_id = m.video_id
""")
spark.sql(f"""
    MERGE INTO {FACT} AS t
    USING fact_mention_src AS s ON t.video_key = s.video_key AND t.restaurant_id = s.restaurant_id
    WHEN MATCHED AND NOT (t.mention_weight <=> s.mention_weight AND t.mention_source <=> s.mention_source
                          AND t.mention_count <=> s.mention_count) THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
f = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT video_key) AS videos, COUNT(DISTINCT restaurant_id) AS restaurants FROM {FACT}").collect()[0]
print(f"✅ {FACT}: {f['n']} mentions, {f['videos']} videos, {f['restaurants']} restaurants")
print("\n--- dim_restaurant + fact_video_restaurant_mention complete ---")

In [ ]:
# --- dim_google_review: one row per Google review (SCD 1, full rebuild; empty until reviews are read) ---
# Was fact_google_review until 2026-11-02: <= 5 'most relevant' reviews per place are a sample to show, not a measure
if spark.catalog.tableExists(f"{GOLD}.fact_google_review"):
    spark.sql(f"DROP TABLE {GOLD}.fact_google_review")
    print(f"🗑️ dropped {GOLD}.fact_google_review (replaced by dim_google_review)")
if spark.catalog.tableExists(REVIEWS):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {REVIEW_DIM}
        COMMENT 'Google review dimension (SCD 1): 1 review (Place Details, <= 5 most relevant per read, accumulated). review_key = xxhash64(review_id), -1 = Unknown; restaurant_id = unknown when no restaurant. Descriptive only: stars are a biased sample, rating measures are in fact_place_rating_snapshot.'
        AS
        SELECT
            xxhash64(r.review_id)                                                   AS review_key,
            r.review_id,
            COALESCE(d.restaurant_id, 'unknown')                                    AS restaurant_id,
            COALESCE(dd.date_key, -1)                                               AS publish_date_key,
            CAST(r.rating AS TINYINT)                                               AS rating,
            r.review_rank,
            r.review_text,
            r.review_language,
            r.original_text,
            r.original_language,
            r.author_name,
            r.publish_time,
            r.relative_time,
            r.google_maps_uri,
            r.first_seen_at,
            r.last_seen_at,
            current_timestamp()                                                     AS etl_date
        FROM {REVIEWS} r
        LEFT JOIN (SELECT place_id, min(restaurant_id) AS restaurant_id FROM {DIM}
                   WHERE is_current AND restaurant_id <> 'unknown' AND place_id IS NOT NULL GROUP BY place_id) d
          ON d.place_id = r.place_id
        LEFT JOIN {GOLD}.dim_date dd
          ON dd.date_key = CAST(date_format(from_utc_timestamp(r.publish_time, 'Asia/Ho_Chi_Minh'), 'yyyyMMdd') AS INT)
    """)
    g = spark.sql(f"""
        SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id) AS restaurants, ROUND(AVG(rating), 2) AS avg_rating,
               COUNT_IF(restaurant_id = 'unknown') AS no_restaurant
        FROM {REVIEW_DIM} WHERE review_key <> -1""").collect()[0]
    spark.sql(f"""
        INSERT INTO {REVIEW_DIM} (review_key, restaurant_id, publish_date_key, review_text, etl_date)
        SELECT -1, 'unknown', -1, 'Unknown', current_timestamp()
        WHERE NOT EXISTS (SELECT 1 FROM {REVIEW_DIM} WHERE review_key = -1)
    """)
    print(f"✅ {REVIEW_DIM}: {g['n']} reviews of {g['restaurants']} restaurants · avg {g['avg_rating']} · "
          f"no current restaurant: {g['no_restaurant']}")
else:
    print(f"{REVIEWS} does not exist yet (stg/restaurant_place_details has not read reviews): {REVIEW_DIM} not built")
print("\n--- dim_google_review complete ---")

In [ ]:
# --- dim_dish (SCD 1, full rebuild): dishes the reviews name, child of dim_restaurant ---
DISH_CATEGORIES = [   # first match on the unaccented dish name
    ("Đồ uống", r"\\b(tra|ca phe|cafe|coffee|nuoc|sinh to|sua|bia|ruou|soda|tra sua|matcha|cacao|nuoc ep)\\b"),
    ("Tráng miệng", r"\\b(che|kem|flan|rau cau|tau hu|banh flan|yaourt|sua chua|banh ngot|tiramisu)\\b"),
    ("Món nước", r"\\b(pho|bun|hu tieu|mi|banh canh|chao|lau|sup|mien|banh da|cao lau|nui)\\b"),
    ("Cơm / xôi / bánh mì", r"\\b(com|xoi|banh mi|com tam|com chien)\\b"),
    ("Hải sản", r"\\b(oc|tom|cua|ghe|so|muc|ca|bach tuoc|hau|ngheu|hen|tom hum)\\b"),
    ("Nướng / chiên", r"\\b(nuong|chien|quay|bbq|xien|ran)\\b"),
    ("Ăn vặt", r"\\b(banh trang|goi cuon|cha gio|nem|bot chien|banh xeo|banh khot|xien que|banh bot loc)\\b"),
]
category_rule = " ".join(f"WHEN dish_plain RLIKE '{rx}' THEN '{cat}'" for cat, rx in DISH_CATEGORIES)
spark.sql(f"""
    CREATE OR REPLACE TABLE {DISH}
    COMMENT 'Dish dimension (SCD 1, full rebuild): 1 restaurant_id x 1 dish named in the YouTube reviews (stg_restaurant_mentions.dishes). Key: restaurant_id + dish_name_normalized. Price = median said in the videos.'
    AS
    WITH d AS (
        SELECT m.restaurant_id, normalize_keyword(x.name) AS dish_name_normalized, trim(x.name) AS dish,
               CASE WHEN x.price_vnd BETWEEN 5000 AND 5000000 THEN x.price_vnd END AS price_vnd,
               coalesce(x.is_signature, false) AS is_signature, m.video_id, m.extracted_at
        FROM mentions_keyed m
        LATERAL VIEW explode({'m.dishes' if has_dishes else 'CAST(NULL AS ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>)'}) t AS x
        WHERE m.restaurant_id IS NOT NULL AND length(trim(x.name)) > 1
    ),
    g AS (
        SELECT restaurant_id, dish_name_normalized,
               max_by(dish, length(dish))                    AS dish_name,
               lower(unaccent_vi(dish_name_normalized))      AS dish_plain,
               CAST(percentile_approx(price_vnd, 0.5) AS BIGINT) AS price_vnd,
               MIN(price_vnd)                                AS price_min_vnd,
               MAX(price_vnd)                                AS price_max_vnd,
               bool_or(is_signature)                         AS is_signature,
               COUNT_IF(is_signature)                        AS signature_votes,
               COUNT(DISTINCT video_id)                      AS video_count,
               MIN(extracted_at)                             AS first_seen_at,
               MAX(extracted_at)                             AS last_seen_at
        FROM d GROUP BY restaurant_id, dish_name_normalized
    )
    SELECT g.restaurant_id, g.dish_name_normalized, g.dish_name,
           CASE {category_rule} ELSE 'Khác' END AS dish_category,
           g.price_vnd, g.price_min_vnd, g.price_max_vnd, g.is_signature, CAST(g.signature_votes AS INT) AS signature_votes,
           CAST(g.video_count AS INT) AS video_count, g.first_seen_at, g.last_seen_at,
           'video' AS source, current_timestamp() AS etl_date
    FROM g
    WHERE EXISTS (SELECT 1 FROM {DIM} r WHERE r.restaurant_id = g.restaurant_id)
""")
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id) AS restaurants, COUNT(price_vnd) AS priced,
           COUNT_IF(is_signature) AS signature, COUNT(*) - COUNT(DISTINCT restaurant_id, dish_name_normalized) AS dup
    FROM {DISH}""").collect()[0]
print(f"✅ {DISH}: {s['n']} dishes of {s['restaurants']} restaurants · {s['priced']} with a price · "
      f"{s['signature']} signature · grain: " + ("✅" if s["dup"] == 0 else f"❌ {s['dup']} duplicates"))
display(spark.sql(f"SELECT dish_category, COUNT(*) AS dishes, round(AVG(price_vnd)) AS avg_price FROM {DISH} GROUP BY 1 ORDER BY 2 DESC"))
print("\n--- dim_dish complete ---")